# Neural Networks with PyTorch

## A Hands-On Introduction to Deep Learning

## What Are Neural Networks?

Neural networks are computational systems inspired by the human brain's structure and function.

### Key Concepts:

- **Building Blocks**: Artificial neurons (nodes) organized in layers
- **Learning**: Networks learn patterns from data through training
- **Applications**: Image recognition, natural language processing, game playing, and more

### From Simple to Complex:

1. **Logistic Regression**: Single-layer network, linear decision boundaries
2. **Multi-Layer Perceptron (MLP)**: Adds hidden layers, learns non-linear patterns
3. **Deep Neural Networks**: Many layers, can approximate any function (Universal Approximation Theorem)

### Why PyTorch?

PyTorch is an open-source deep learning framework that offers:
- **Dynamic Computation Graphs**: Flexibility in model architecture
- **Pythonic API**: Intuitive syntax similar to NumPy
- **GPU Acceleration**: Seamless CUDA integration for fast training
- **Strong Community**: Extensive documentation and active research community

## Importing Required Libraries

In [ ]:
# Standard data science libraries
import numpy as np      # Numerical computations and array operations
import pandas as pd     # Data manipulation and CSV file handling
import matplotlib.pyplot as plt  # Data visualization

# PyTorch libraries for deep learning
import torch            # Core PyTorch library
import torch.nn as nn   # Neural network modules (layers, loss functions)
import torch.optim as optim  # Optimization algorithms
from torch.utils.data import Dataset, DataLoader  # Data handling utilities

# Set random seed for reproducibility
# This ensures we get the same results every time we run the code
torch.manual_seed(42)
np.random.seed(42)

print("✓ Libraries imported successfully")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

## Understanding the MNIST Dataset

### What is MNIST?

**MNIST** (Modified National Institute of Standards and Technology) is the "Hello World" of machine learning:

- **Content**: 70,000 handwritten digits (0-9)
- **Training Set**: 60,000 images for training
- **Test Set**: 10,000 images for evaluation
- **Image Format**: 28 × 28 pixel grayscale images
- **Labels**: Integer values 0-9 representing the digit

### Why Use Local CSV Files?

Instead of downloading from the internet, we'll use local CSV files. This approach:

- ✅ **Works Offline**: No internet connection required
- ✅ **Faster Access**: Direct file I/O instead of HTTP download
- ✅ **Reproducible**: Same data every time
- ✅ **Privacy**: Data stays on your local machine

## Loading the MNIST Dataset from Local CSV Files

In [ ]:
# Load MNIST data from local CSV files
# The CSV format: first column = label (0-9), remaining 784 columns = pixel values (28×28)

print("📂 Loading training data from mnist_train.csv...")
train_df = pd.read_csv('mnist_train.csv', header=None)
print(f"   ✓ Loaded {len(train_df):,} training samples")

print("\n📂 Loading test data from mnist_test.csv...")
test_df = pd.read_csv('mnist_test.csv', header=None)
print(f"   ✓ Loaded {len(test_df):,} test samples")

# Extract labels (first column) and pixel values (remaining columns)
y_train = train_df.iloc[:, 0].values      # Training labels: 0-9
X_train = train_df.iloc[:, 1:].values    # Training images: 784 pixels

y_test = test_df.iloc[:, 0].values        # Test labels
X_test = test_df.iloc[:, 1:].values      # Test images

print("\n📊 Dataset Information:")
print(f"   Training set: {X_train.shape[0]} samples × {X_train.shape[1]} features")
print(f"   Test set: {X_test.shape[0]} samples × {X_test.shape[1]} features")
print(f"   Image dimensions: 28 × 28 = 784 pixels")

## Exploring the Dataset: Label Distribution

In [ ]:
# Analyze the distribution of labels in the dataset
# This helps us understand if the dataset is balanced

# Training set distribution
unique_train, counts_train = np.unique(y_train, return_counts=True)
print("📈 Training Set Label Distribution:")
for digit, count in sorted(zip(unique_train, counts_train)):
    print(f"   Digit {digit}: {count:5,} samples ({count/len(y_train)*100:4.1f}%)")

# Test set distribution
unique_test, counts_test = np.unique(y_test, return_counts=True)
print("\n📈 Test Set Label Distribution:")
for digit, count in sorted(zip(unique_test, counts_test)):
    print(f"   Digit {digit}: {count:4,} samples ({count/len(y_test)*100:4.1f}%)")

# Check if the dataset is balanced
print("\n✅ Dataset Quality Checks:")
print(f"   Training: min={min(counts_train)}, max={max(counts_train)} samples per class")
print(f"   Test: min={min(counts_test)}, max={max(counts_test)} samples per class")
print(f"   Status: {'Well-balanced ✓' if max(counts_train) - min(counts_train) < 1000 else 'Imbalanced ⚠'}")

## Visualizing Sample Images

In [ ]:
# Display a grid of random sample images from the training set
# Visualization helps us understand the data we're working with

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
fig.suptitle('Random Sample of 25 MNIST Training Images', fontsize=16, fontweight='bold')

# Randomly select 25 samples
np.random.seed(42)  # For reproducible visualization
indices = np.random.choice(len(X_train), 25, replace=False)

for idx, ax in zip(indices, axes.flat):
    # Reshape 784-pixel vector to 28×28 image
    image = X_train[idx].reshape(28, 28)
    
    # Display image
    ax.imshow(image, cmap='gray', interpolation='nearest')
    ax.set_title(f'Label: {y_train[idx]}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print(f"\n🖼️  Displayed 25 random samples from the training set")

## Data Preprocessing: Why It Matters

### Why Do We Need Preprocessing?

Neural networks work best with normalized data. Raw pixel values (0-255) can cause:

- **Slow convergence**: Large values make optimization harder
- **Numerical instability**: Risk of overflow/underflow
- **Uneven feature scales**: Some features dominate others

### Our Preprocessing Steps:

1. **Normalization**: Scale pixel values to [0, 1] range
2. **Tensor Conversion**: Convert NumPy arrays to PyTorch tensors
3. **Data Type**: Use float32 for features, long for labels

In [ ]:
# Data preprocessing pipeline

print("🔧 Step 1: Normalizing pixel values")
# Original pixel values are 0-255 (8-bit grayscale)
# We normalize to 0.0-1.0 for better neural network training
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
print(f"   ✓ Training range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"   ✓ Test range: [{X_test.min():.3f}, {X_test.max():.3f}]")

print("\n🔧 Step 2: Converting to PyTorch tensors")
# Convert features to float32 tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)

# Convert labels to long integers (required for CrossEntropyLoss)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

print(f"   ✓ X_train_tensor shape: {X_train_tensor.shape}")
print(f"   ✓ y_train_tensor shape: {y_train_tensor.shape}")

print("\n📊 Final Data Shapes:")
print(f"   Training: {X_train_tensor.shape[0]} samples, {X_train_tensor.shape[1]} features")
print(f"   Test: {X_test_tensor.shape[0]} samples, {X_test_tensor.shape[1]} features")
print(f"   Classes: 10 (digits 0-9)")

## Creating Custom PyTorch Dataset

### What is a PyTorch Dataset?

A PyTorch `Dataset` is an abstract class that allows us to:

- **Encapsulate data**: Store features and labels together
- **Enable batching**: Process data in mini-batches for efficient training
- **Apply transformations**: Data augmentation, normalization, etc.

We create a custom `MNISTDataset` class that inherits from `torch.utils.data.Dataset`.

In [ ]:
# Define a custom Dataset class for MNIST
class MNISTDataset(Dataset):
    """
    Custom PyTorch Dataset for MNIST handwritten digits.
    
    Args:
        X: Feature tensor (normalized pixel values)
        y: Label tensor (digit class 0-9)
    """
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        """Return the total number of samples."""
        return len(self.X)
    
    def __getitem__(self, idx):
        """
        Get a single sample by index.
        Returns: (features, label) tuple
        """
        return self.X[idx], self.y[idx]

# Create dataset instances
print("📦 Creating PyTorch Datasets...")
train_dataset = MNISTDataset(X_train_tensor, y_train_tensor)
test_dataset = MNISTDataset(X_test_tensor, y_test_tensor)

print(f"   ✓ Training dataset: {len(train_dataset):,} samples")
print(f"   ✓ Test dataset: {len(test_dataset):,} samples")

# Test the dataset
sample_features, sample_label = train_dataset[0]
print(f"\n🎯 Sample data point:")
print(f"   Features shape: {sample_features.shape}")
print(f"   Label: {sample_label.item()}")

## Creating DataLoaders for Batching

### Why Use DataLoaders?

PyTorch `DataLoader` provides:

- **Batching**: Group samples into mini-batches (e.g., 128 samples at a time)
- **Shuffling**: Randomize order each epoch (prevents learning order patterns)
- **Parallel Loading**: Load data in background for faster training

### Batch Size Selection:

- **Larger batches** (256-512): Faster training, requires more memory
- **Smaller batches** (32-64): Better generalization, more updates per epoch
- **Sweet spot** (128): Good balance used in this notebook

In [ ]:
# Create DataLoaders for batching
batch_size = 128  # Number of samples per batch

print(f"🔄 Creating DataLoaders (batch_size={batch_size})...")

# Training DataLoader: shuffle=True to randomize order each epoch
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0  # Set to 0 for compatibility
)

# Test DataLoader: shuffle=False (no need to randomize test data)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"   ✓ Training batches: {len(train_loader)}")
print(f"   ✓ Test batches: {len(test_loader)}")

# Verify batch shapes
for batch_X, batch_y in train_loader:
    print(f"\n📐 Batch shapes:")
    print(f"   Features: {batch_X.shape} (batch_size, features)")
    print(f"   Labels: {batch_y.shape} (batch_size)")
    break  # Only show first batch

# Building the Neural Network Model

## Model Architecture Design

### Our Network Structure:

```
Input (784) → Dense(256) → ReLU → Dropout(0.45) →
              Dense(256) → ReLU → Dropout(0.45) →
              Dense(10) → Softmax
```

### Layer-by-Layer Breakdown:

1. **Input Layer**: 784 neurons (28×28 pixels)
2. **Hidden Layer 1**: 256 neurons with ReLU activation
3. **Dropout 1**: 45% dropout (prevents overfitting)
4. **Hidden Layer 2**: 256 neurons with ReLU activation
5. **Dropout 2**: 45% dropout
6. **Output Layer**: 10 neurons (one per digit class)

In [ ]:
# Define the neural network architecture
class MNISTClassifier(nn.Module):
    """
    Multi-layer perceptron for MNIST digit classification.
    
    Architecture:
        Input (784) → Dense(256) → ReLU → Dropout(0.45) →
                      Dense(256) → ReLU → Dropout(0.45) →
                      Dense(10) → Output
    """
    
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, dropout=0.45):
        super(MNISTClassifier, self).__init__()
        
        # Layer 1: Input to first hidden layer
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        # Layer 2: First to second hidden layer
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        # Output layer
        self.fc3 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, 784)
            
        Returns:
            Output logits of shape (batch_size, 10)
        """
        # Layer 1
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        # Layer 2
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        # Output layer (no activation - CrossEntropyLoss handles Softmax)
        x = self.fc3(x)
        
        return x

# Create the model instance
print("🏗️  Creating MNIST Classifier Model...")
model = MNISTClassifier(input_size=784, hidden_size=256, num_classes=10, dropout=0.45)

# Display model architecture
print("\n📋 Model Architecture:")
print(model)

# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Parameter Count:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

## Defining Loss Function and Optimizer

### Loss Function: Cross-Entropy Loss

For multi-class classification, we use **Cross-Entropy Loss** (also called log loss):

- Measures the difference between predicted probability distribution and true distribution
- Combines `LogSoftmax` and `NLLLoss` in one step
- Lower loss = better predictions

### Optimizer: Adam

**Adam** (Adaptive Moment Estimation) is an advanced optimization algorithm:

- Combines benefits of AdaGrad and RMSProp
- Adapts learning rate for each parameter
- Generally faster convergence than standard SGD
- Default learning rate of 0.001 works well for most cases

In [ ]:
# Define loss function and optimizer

# Loss function: Cross-Entropy Loss for multi-class classification
# This combines LogSoftmax and Negative Log Likelihood Loss
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam with learning rate 0.001
# Adam adapts learning rates per-parameter for efficient optimization
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("⚙️  Training Configuration:")
print(f"   Loss Function: CrossEntropyLoss")
print(f"   Optimizer: Adam")
print(f"   Learning Rate: {learning_rate}")
print(f"   Batch Size: {batch_size}")

# Optional: Learning rate scheduler (reduces LR when loss plateaus)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)
print(f"\n📉 Learning Rate Scheduler: ReduceLROnPlateau")
print(f"   Factor: 0.5, Patience: 3 epochs")

## Training the Model

### Training Process:

1. **Forward Pass**: Input data goes through the network to get predictions
2. **Calculate Loss**: Compare predictions with true labels
3. **Backward Pass**: Compute gradients (backpropagation)
4. **Update Weights**: Optimizer adjusts parameters to reduce loss
5. **Repeat**: Process all batches for multiple epochs

### Epochs and Batches:

- **Epoch**: One complete pass through the entire training dataset
- **Batch**: A subset of data processed together (memory efficiency)
- **Iteration**: One forward/backward pass on a single batch

In [ ]:
# Training parameters
num_epochs = 20  # Number of times to iterate through the entire dataset
print_every = 100  # Print progress every N batches

# Lists to store metrics for plotting
train_losses = []
train_accuracies = []

print(f"🚀 Starting Training for {num_epochs} epochs...")
print("=" * 60)

# Training loop
for epoch in range(num_epochs):
    model.train()  # Set model to training mode (enables dropout)
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Iterate through batches
    for batch_idx, (data, target) in enumerate(train_loader):
        # Zero the gradients from previous iteration
        optimizer.zero_grad()
        
        # Forward pass: get predictions
        outputs = model(data)
        
        # Calculate loss
        loss = criterion(outputs, target)
        
        # Backward pass: compute gradients
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        # Print progress
        if (batch_idx + 1) % print_every == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], "
                  f"Batch [{batch_idx+1}/{len(train_loader)}], "
                  f"Loss: {loss.item():.4f}")
    
    # Calculate epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    # Print epoch summary
    print(f"\n📊 Epoch [{epoch+1}/{num_epochs}] Summary:")
    print(f"   Average Loss: {epoch_loss:.4f}")
    print(f"   Accuracy: {epoch_acc:.2f}%")
    print(f"   Correct: {correct}/{total}")
    print("-" * 60)
    
    # Update learning rate scheduler
    scheduler.step(epoch_loss)

print(f"\n✅ Training Complete!")

## Visualizing Training Progress

Let's plot the training loss and accuracy over epochs to see how our model learned:


In [ ]:
# Plot training metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot Loss
ax1.plot(range(1, num_epochs + 1), train_losses, 'b-', linewidth=2, marker='o')
ax1.set_title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, num_epochs + 1))

# Plot Accuracy
ax2.plot(range(1, num_epochs + 1), train_accuracies, 'g-', linewidth=2, marker='o')
ax2.set_title('Training Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, num_epochs + 1))

plt.tight_layout()
plt.show()

print(f"\n📈 Final Training Accuracy: {train_accuracies[-1]:.2f}%")
print(f"📉 Final Training Loss: {train_losses[-1]:.4f}")

# Evaluating the Model

## Testing on the Test Set

Now let's evaluate our trained model on the test set to see how well it generalizes to unseen data:


In [ ]:
# Evaluation mode (disables dropout for consistent predictions)
model.eval()

# Variables to track test performance
test_correct = 0
test_total = 0
test_loss = 0.0

# Disable gradient calculation for efficiency (no backprop during evaluation)
with torch.no_grad():
    for data, target in test_loader:
        # Forward pass
        outputs = model(data)
        
        # Calculate loss
        loss = criterion(outputs, target)
        test_loss += loss.item()
        
        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        
        # Statistics
        test_total += target.size(0)
        test_correct += (predicted == target).sum().item()

# Calculate final test metrics
test_accuracy = 100 * test_correct / test_total
test_loss_avg = test_loss / len(test_loader)

print("🧪 Model Evaluation on Test Set:")
print("=" * 50)
print(f"   Test Loss: {test_loss_avg:.4f}")
print(f"   Test Accuracy: {test_accuracy:.2f}%")
print(f"   Correct Predictions: {test_correct:,} / {test_total:,}")
print(f"   Error Rate: {100 - test_accuracy:.2f}%")

if test_accuracy >= 95:
    print(f"\n🎉 Excellent! Accuracy >= 95%")
elif test_accuracy >= 90:
    print(f"\n👍 Good! Accuracy >= 90%")
else:
    print(f"\n⚠️  Accuracy below 90% - may need more training or architecture tuning")

## Making Predictions on Sample Images

Let's visualize some actual predictions to see our model in action:


In [ ]:
# Get some test samples for visualization
model.eval()
with torch.no_grad():
    # Get first batch of test data
    data, target = next(iter(test_loader))
    outputs = model(data)
    _, predicted = torch.max(outputs, 1)
    
    # Convert tensors to numpy for visualization
    images = data.numpy()
    true_labels = target.numpy()
    pred_labels = predicted.numpy()

# Create visualization
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Model Predictions on Test Images', fontsize=16, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    if idx < len(images):
        # Reshape and display image
        image = images[idx].reshape(28, 28)
        ax.imshow(image, cmap='gray')
        
        # Color code: green for correct, red for incorrect
        color = 'green' if true_labels[idx] == pred_labels[idx] else 'red'
        ax.set_title(f'True: {true_labels[idx]}\nPred: {pred_labels[idx]}', 
                    color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

# Calculate accuracy on these samples
sample_correct = sum(true_labels[:15] == pred_labels[:15])
print(f"\n📊 Sample Accuracy: {sample_correct}/15 ({100*sample_correct/15:.1f}%)")

## Saving the Trained Model

We can save the model for later use without retraining:


In [ ]:
# Save the trained model
model_save_path = 'mnist_pytorch_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
    'test_accuracy': test_accuracy,
    'model_architecture': model.__class__.__name__,
}, model_save_path)

print(f"💾 Model saved to: {model_save_path}")
print(f"\n📦 Saved content:")
print(f"   - Model weights (state_dict)")
print(f"   - Optimizer state")
print(f"   - Training epochs: {num_epochs}")
print(f"   - Test accuracy: {test_accuracy:.2f}%")

print(f"\n📝 To load the model later:")
print(f"   model = MNISTClassifier()")
print(f"   checkpoint = torch.load('{model_save_path}')")
print(f"   model.load_state_dict(checkpoint['model_state_dict'])")

# Summary and Key Takeaways

## What We Learned:

1. **Data Loading**: Reading MNIST from local CSV files instead of downloading
2. **PyTorch Basics**: Tensors, Datasets, DataLoaders
3. **Model Building**: Creating neural networks with `nn.Module`
4. **Training Loop**: Forward pass → Loss → Backprop → Update
5. **Evaluation**: Testing model performance on unseen data

## Key Differences: TensorFlow vs PyTorch

| Aspect | TensorFlow/Keras | PyTorch |
|--------|-----------------|---------|
| Graph | Static | Dynamic |
| Training Loop | `model.fit()` | Manual loop |
| Model Definition | `Sequential()` | `nn.Module` class |
| Data | Built-in loaders | Custom `Dataset` class |
| Flexibility | High-level | More control |

## Final Results:

- **Model**: 3-layer MLP with 269,322 parameters
- **Training Accuracy**: ~98-99%
- **Test Accuracy**: ~96-98% (typical for this architecture)

🎉 **Congratulations! You've built and trained your first neural network with PyTorch!**